In [ ]:
import pandas as pd
import altair as alt

# Charts that show the breakdown of a total

In [ ]:
austria_election = pd.DataFrame({
    "category": ["ÖVP", "SPÖ", "FPÖ", "Grüne", "NEOS", "Bierpartei", "Others"],
    "value": [26.3,  21.1, 28.8,  8.2, 9.1, 2.0, 4.5]
})
austria_election

## Donut Chart

In [ ]:
alt.Chart(austria_election).mark_arc(innerRadius=50).encode(
    theta= alt.Theta("value:Q"),
    color=alt.Color("category:N")
)

## Pie Chart

In [ ]:
base = alt.Chart(austria_election).encode(
    alt.Theta("value:Q").stack(True),
    alt.Color("category:N").legend(None)
)

pie = base.mark_arc(outerRadius=120)
text = base.mark_text(radius=140, size=15).encode(text="category:N")

pie + text

# Chart for Temporal Data

In [ ]:
from vega_datasets import data
stocks = data.stocks()
stocks

In [ ]:
alt.Chart(stocks).mark_line().encode(
    x='date:T',
    y='price:Q',
    color='symbol'
)

## Area Chart

In [ ]:
alt.Chart(stocks).transform_filter(
    'datum.symbol==="GOOG"'
).mark_area().encode(
    alt.X('date:T'),
    alt.Y('price:Q')
)

## Stacked Area Chart

In [ ]:
alt.Chart(stocks).mark_area().encode(
    x="date:T",
    y="price:Q",
    color="symbol:N"
)


## Slope Chart

In [ ]:
first_last_stocks = stocks.groupby('symbol').apply(lambda x: pd.concat([x.head(1), x.tail(1)])).reset_index(drop=True)
first_last_stocks = first_last_stocks[first_last_stocks['symbol'] != 'GOOG']
first_last_stocks

In [ ]:
alt.Chart(first_last_stocks).mark_line().encode(
    x='date:O',
    y='price',
    color='symbol'
)

In [ ]:
first_last_stocks['year'] = pd.DatetimeIndex(first_last_stocks['date']).year
first_last_stocks

In [ ]:
alt.Chart(first_last_stocks).mark_line().encode(
    x='year:O',
    y='price',
    color='symbol'
).properties(
    width=300
)

# Charts for Spatial Data

In [ ]:
airports = data.airports()
airports.head()

The chart is quite similar to a standard scatterplot, with a few differences:

We specify the encodings “latitude” and “longitude” instead of “x” and “y” and we specify a projection to be used for the data. For data covering only the United States, the projection “albersUsa” is useful:

In [ ]:
alt.Chart(airports).mark_circle().encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    size=alt.value(7),
    tooltip='name'
).project(
    "albersUsa"
).properties(
    width=400,
    height=300
)

All available projections are shown in the [Vega Documentation](https://vega.github.io/vega/docs/projections/) and in the [Vega-Lite Docuemtantion](https://vega.github.io/vega-lite/docs/projection.html).

## Background maps

If a geographical map with states and country borders is to be displayed, then a geographical shape file must be loaded into Altair and the ``geoshape`` marker must be used:

In [ ]:
states = alt.topo_feature(data.us_10m.url, feature='states')

alt.Chart(states).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project('albersUsa').properties(
    width=500,
    height=300
)

And here are the world's national borders:

In [ ]:
countries = alt.topo_feature(data.world_110m.url, 'countries')

alt.Chart(countries).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project(
    "equirectangular"
).properties(
    width=500,
    height=300
)

You can see what happens if you try other projection types, e.g. ``equirectangular``, ``mercator``, ``orthographic``, ``albers``, oder ``gnomonic``

### Different projections:

In [ ]:
source = alt.topo_feature(data.world_110m.url, 'countries')

base = alt.Chart(source).mark_geoshape(
    fill='#666666',
    stroke='white'
).properties(
    width=300,
    height=180
)

projections = ['equirectangular', 'mercator', 'orthographic', 'gnomonic'] # go through different projects (see Altair docu)
charts = [base.project(proj).properties(title=proj)
          for proj in projections]

alt.concat(*charts, columns=2)

# Using maps from other countries

In [ ]:
#!pip install geopandas gpdvega

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon # in case points and polygons are needed

### Shapefiles

* .shp - shapefile that is actually geometry like states
* .dbf - database with addional infos outside the shape
* .prj - projection file with shape of earth etc.
* .shx - index file (makes it faster)

Austria: https://www.esri-austria.at/service/open-data-und-projektionen-fuer-oesterreich/

https://data-synergis.opendata.arcgis.com/datasets/SynerGIS::bundesl%C3%A4ndergrenzen-150-000-stand-2017/explore?location=47.665927%2C13.346488%2C6.99


In [ ]:
# when reading the shape file the other files in the folder will be read automatically
austria = gpd.read_file("shapefiles_austria/Bundeslaender_50.shp")
austria

In [ ]:
base_austria = alt.Chart(austria).mark_geoshape(
    fill='lightgrey',
    stroke='#AAAAAA'
).properties(
    width=300,
    height=180
)
base_austria

**Filtern einzelner Bundesländer:**

In [ ]:
alt.Chart(austria[austria.SHAPE_Area > 1.5]).mark_geoshape(
    fill='darkblue',
    stroke='#AAAAAA'
).properties(
    width=300,
    height=180
)

Auch hier können verschiedene Projektionen angewandt werden:

In [ ]:
alt.Chart(austria).mark_geoshape(
    fill="#EEEEEE", stroke="darkgray"
).project('conicEqualArea').properties(
    width=500,
    height=300
)

In [ ]:
alt.Chart(austria).mark_geoshape(
    fill="#EEEEEE", stroke="darkgray"
).project('conicConformal')

## Dots on a Map

In Altair, two charts can be placed on top of each other using '+'

In [ ]:
states = alt.topo_feature(data.us_10m.url, feature='states')
airports = data.airports()

background = alt.Chart(states).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project('albersUsa').properties(
    width=500,
    height=300
)

points = alt.Chart(airports).mark_circle().encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    size=alt.value(8),
    tooltip='name'
)

background + points

### Load data set from Austria and display on map:

**Example Data Set:** Police stations in Austria

https://www.data.gv.at/katalog/dataset/4e5c9414-2886-462e-b19f-a598ad065522#resources

In [ ]:
police = pd.read_csv('policestations_austria.csv', sep=";")
police

In [ ]:
police.info()

Longitude and latitude are still strings and still need to be converted to floats:

In [ ]:
police["X"] = police["X"].str.replace(',', '.').astype(float)
police["Y"] = police["Y"].str.replace(',', '.').astype(float)

In [ ]:
alt.Chart(police).mark_circle(opacity=0.7).encode(
    # point latitude & longitude to coordinates in X & Y columns
    longitude='X:Q',
    latitude='Y:Q',
    tooltip=['ArtLang', 'Bezirk'],
    color=alt.Color('Land:N', scale=alt.Scale(scheme='dark2')) #color scales: https://vega.github.io/vega/docs/schemes/
)

Arrange points on map:

In [ ]:
# 1.  mark_geoshape transparently uses the geometry column
basemap = alt.Chart(austria).mark_geoshape(
    # add some styling to reduce the salience of the basemap
    fill="#444444", stroke="lightgray"
).project('conicConformal').properties(
    width=500,
    height=300
)

# 2.  we use mark_circle to have more control over visual variables
markers = alt.Chart(police).mark_circle(opacity=0.6).encode(
    longitude='X:Q',
    latitude='Y:Q',
    size=alt.value(12),
    tooltip=['ArtLang', 'Bezirk'],
    color=alt.Color('Land:N', scale=alt.Scale(scheme='pastel1')) #color scales: https://vega.github.io/vega/docs/schemes/
)

# combine the two layers
basemap + markers

## Aggregated police stations

In [ ]:
df_police = police[['X', 'Y', 'Land']]
df_police

In [ ]:
df_agg = df_police.groupby('Land').agg({'Land':'count', 'X':'mean', 'Y':'mean'})
df_agg = df_agg.rename(columns={'Land':'stations'}).reset_index()
df_agg

## Symbol Map

In [ ]:
basemap = alt.Chart(austria).mark_geoshape(
    fill="#AAAAAA", stroke="lightgray"
).project('conicConformal').properties(
    width=500,
    height=300
)

markers = alt.Chart(df_agg).mark_circle(opacity=0.6).encode(
    longitude='X:Q',
    latitude='Y:Q',
    size=alt.Size('stations:Q'),
    color=alt.value('steelblue'),
    tooltip=['Land:N','stations:Q']
)

basemap + markers

## Choropleth Map

In [ ]:
austria_police = austria.merge(df_agg, left_on='BL', right_on='Land')
austria_police

In [ ]:
alt.Chart(austria_police).mark_geoshape().encode(
     color='stations:Q',
     tooltip=['Land:N','stations:Q']
).project('conicConformal').properties(
    width=500,
    height=300
)

###### 🚀 Exercise 1 - Dot Density Map:
Show the petrol stations on the Austrian map. Take only the petrol stations from a single day (e.g. the last day). Use the tooltip to display the name of the petrol station and encode the super or diesel prices using an ordered color scale.

###### 🚀 Exercise 2 - Choropleth Map or Proportional Symbol Map :
Display the petrol stations on the Austrian map in aggregated form (e.g. in a Choropleth Map or Proporional Symbol Map). Show either the diesel or Super95 price on the map.

How are the prices distributed on average in Austria? Are there differences between east and west? (Attention: all values with 0.0 should be removed beforehand, otherwise the prices will be distorted).